# Understanding Rubin source injection and forced photometry

This guided notebook follows **one controlled source through one real Rubin image**. It is an inspection and teaching notebook: it reads Rubin data, injects only into an in-memory clone, and does not modify production datasets.

Run it on the Rubin RSP after initializing the stack in the terminal that starts Jupyter:

```bash
setup lsst_distrib
```

The controlled source represents one simulated event at one epoch. In the full science pipeline, event parameters first determine the time-dependent magnification and ideal magnitude:

```text
simulated event
      |
      v
event parameters
      |
      v
magnification(t)
      |
      v
ideal magnitude
      |
      v
source selected for image
      |
      v
injection catalog
      |
      v
PhotoCalib / PSF / WCS
      |
      v
VisitInjectTask
      |
      v
injected exposure
      |
      v
ForcedMeasurementTask
      |
      v
photometry row
```

The release-aware software path is:

```text
release name
    -> get_data_release(...)
    -> DataReleaseConfig
    -> LSSTData
    -> ImageContext
    -> ImageProcessor
```

- `DataReleaseConfig` owns release-specific repository, collection, dataset, table, column, and data-ID conventions.
- `LSSTData` performs Butler/TAP access using that resolved configuration.
- `ImageContext` describes one generic image without loading its pixels.
- `ImageProcessor` owns one image-level scientific operation. The current implementation injects sources and performs forced photometry in memory.

## 1. Configuration and repository map

**What are we trying to understand?** Where the maintained implementation and validation record live.

**Expected:** changing `RELEASE_NAME` from `"DP1"` to `"DP0.2"` is the only release-selection change. No Butler collection or dataset name is configured here.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# One-line release selection. DataReleaseConfig resolves all concrete Rubin names.
RELEASE_NAME = "DP1"  # Change to "DP0.2" to repeat the walkthrough there.
REQUESTED_MAG = 22.0
INJECTION_NOISE_SEED = 12345

# Locate the repository whether Jupyter starts in the repository root or docs/tutorials.
REPO_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "ulens_lsst" / "image_processors.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

paths = {
    "validation status": REPO_ROOT / "docs/scientific_validation_status.rst",
    "validation methods": REPO_ROOT / "docs/scientific_validation.rst",
    "validation figures/products": REPO_ROOT / "docs/_static/scientific_validation/2026-09-25",
    "ImageContext": REPO_ROOT / "ulens_lsst/image_context.py",
    "LSSTData": REPO_ROOT / "ulens_lsst/lsst_data.py",
    "ImageProcessor": REPO_ROOT / "ulens_lsst/image_processors.py",
    "injection + forced measurement": REPO_ROOT / "ulens_lsst/lsst_tools.py",
    "production PROCESS/schema normalization": REPO_ROOT / "ulens_lsst/image_processing_run.py",
    "controlled validation schema": REPO_ROOT / "ulens_lsst/scientific_validation.py",
    "density validation schema": REPO_ROOT / "ulens_lsst/density_validation.py",
}
pd.Series({name: str(path.relative_to(REPO_ROOT)) for name, path in paths.items()}, name="repository path")

## 2. Resolve one release and one real image

**What are we trying to understand?** Which facts come from the release profile, and which facts identify an individual image.

**Expected:** repository, collection, and image dataset type come from `DataReleaseConfig`. Visit, detector, band, and image identity come from the selected Butler `DatasetRef`.

The next two cells perform bounded registry queries and load one image. They require an initialized RSP environment and may take several seconds.

In [ ]:
from data_release_config import get_data_release
from ulens_lsst.lsst_data import LSSTData

release_config = get_data_release(RELEASE_NAME)
release_fields = {
    "name": release_config.name,
    "rsp_instance": getattr(release_config, "rsp_instance", None),
    "butler_repo": release_config.butler_repo,
    "butler_collections": release_config.butler_collections,
    "image_dataset_types": tuple(release_config.calexp_dataset_types),
    "tap_visit_table": getattr(release_config, "tap_visit_table", None),
    "photometry_method": getattr(release_config, "photometry_method", None),
}
display(pd.Series(release_fields, name="resolved DataReleaseConfig"))

image_data = LSSTData(release_config=release_config, main_path="/tmp")
dataset_type = tuple(release_config.calexp_dataset_types)[0]
collections = release_config.butler_collections

In [ ]:
# Query dimensions before DatasetRefs to keep registry access bounded.
visit_record = next(
    iter(
        image_data.butler.registry.queryDataIds(
            ["visit"], datasets=dataset_type, collections=collections
        )
    ),
    None,
)
if visit_record is None:
    raise RuntimeError(f"No {dataset_type!r} visit is accessible for {RELEASE_NAME}.")

visit_id = int(visit_record["visit"])
data_id_record = next(
    iter(
        image_data.butler.registry.queryDataIds(
            ["visit", "detector"],
            datasets=dataset_type,
            collections=collections,
            where=f"visit={visit_id}",
        )
    ),
    None,
)
image_ref = next(
    iter(
        image_data.butler.registry.queryDatasets(
            dataset_type,
            collections=collections,
            dataId=data_id_record,
            findFirst=True,
        )
    ),
    None,
)
if image_ref is None:
    raise RuntimeError("The selected visit/detector has no accessible image DatasetRef.")

data_id = {key: image_ref.dataId[key] for key in image_ref.datasetType.dimensions.names}
pd.Series(data_id, name="DatasetRef data ID")

In [ ]:
from ulens_lsst.image_context import ImageContext
from ulens_lsst.work_units import make_image_id

image_id = make_image_id(release_config.name, dataset_type, data_id)
context = ImageContext(
    image_id=image_id,
    dataset_type=dataset_type,
    data_id=data_id,
    visit=int(data_id["visit"]),
    detector=int(data_id["detector"]),
    band=str(data_id["band"]),
)
image = image_data.load_image(context)  # Calexp compatibility wrapper around a Rubin ExposureF.
exposure = image.expF

visit_info = exposure.getInfo().getVisitInfo()
observation_mjd_tai = float(visit_info.getDate().get())
context = ImageContext(
    image_id=context.image_id,
    dataset_type=context.dataset_type,
    data_id=context.data_id,
    visit=context.visit,
    detector=context.detector,
    band=context.band,
    mjd=observation_mjd_tai,
)

display(pd.Series({
    "image_id": context.image_id,
    "dataset_type (resolved by DataReleaseConfig)": context.dataset_type,
    "visit": context.visit,
    "detector": context.detector,
    "band": context.band,
    "observation MJD (TAI, from Exposure VisitInfo)": context.mjd,
}, name="image context"))

## 3. Inspect the real Exposure

**What are we trying to understand?** The calibrated image components used by injection and measurement.

- **WCS:** converts the requested sky position to detector pixels.
- **PSF:** renders and measures a point source.
- **PhotoCalib:** converts source magnitude to this image's instrumental flux scale.
- **Variance:** pixel uncertainty model used by measurement.
- **Mask:** pixel-quality planes, including injection-related planes added by Rubin.

We print compact summaries, not full arrays or object representations.

In [ ]:
bbox = exposure.getBBox()
wcs = exposure.getWcs()
psf = exposure.getPsf()
photo_calib = exposure.getPhotoCalib()
masked_image = exposure.getMaskedImage()

psf_shape = psf.computeShape(psf.getAveragePosition())
mask_planes = masked_image.getMask().getMaskPlaneDict()
variance_array = masked_image.getVariance().array

display(pd.Series({
    "dimensions (width, height)": (bbox.getWidth(), bbox.getHeight()),
    "pixel origin (min x, min y)": (bbox.getMinX(), bbox.getMinY()),
    "WCS class": type(wcs).__name__,
    "PSF class": type(psf).__name__,
    "PSF determinant radius (pixels)": float(psf_shape.getDeterminantRadius()),
    "PhotoCalib class": type(photo_calib).__name__,
    "instrumental flux at magnitude zero": float(photo_calib.getInstFluxAtZeroMagnitude()),
    "mask plane count": len(mask_planes),
    "median variance": float(np.nanmedian(variance_array)),
}, name="Exposure summary"))

## 4. Define one controlled, isolated source

**What are we trying to understand?** The visible sky-to-pixel coordinate chain.

We choose the image center, well away from detector boundaries. This checks WCS round-trip consistency. It does **not** fit the injected centroid or establish sub-pixel injection accuracy.

In [ ]:
from lsst.geom import Point2D

requested_pixel = Point2D(float(bbox.getCenterX()), float(bbox.getCenterY()))
requested_sky = wcs.pixelToSky(requested_pixel)
roundtrip_pixel = wcs.skyToPixel(requested_sky)
roundtrip_sky = wcs.pixelToSky(roundtrip_pixel)

source_x = float(roundtrip_pixel.getX())
source_y = float(roundtrip_pixel.getY())
source_ra = requested_sky.getRa().asDegrees()
source_dec = requested_sky.getDec().asDegrees()

display(pd.Series({
    "RA (deg)": source_ra,
    "Dec (deg)": source_dec,
    "requested x (pixel)": requested_pixel.getX(),
    "requested y (pixel)": requested_pixel.getY(),
    "round-trip x difference (pixel)": source_x - requested_pixel.getX(),
    "round-trip y difference (pixel)": source_y - requested_pixel.getY(),
    "round-trip sky separation (arcsec)": requested_sky.separation(roundtrip_sky).asArcseconds(),
    "requested magnitude": REQUESTED_MAG,
    "band": context.band,
}, name="controlled source"))

In [ ]:
# A small view of the pre-injection scene. Non-zero scene flux is allowed and expected.
def cutout(array, x, y, half_size=15):
    ix = int(round(x)) - bbox.getMinX()
    iy = int(round(y)) - bbox.getMinY()
    return array[iy-half_size:iy+half_size+1, ix-half_size:ix+half_size+1]

before_cutout = cutout(masked_image.getImage().array, source_x, source_y)
plt.figure(figsize=(4, 4))
plt.imshow(before_cutout, origin="lower", cmap="gray")
plt.colorbar(label="Instrumental image units")
plt.title("Original image near the controlled position")
plt.xlabel("Cutout x (pixel)")
plt.ylabel("Cutout y (pixel)")
plt.show()

## 5. Follow the magnitude-to-flux chain

**What are we trying to understand?** The exact calibrated flux implied by the requested magnitude and the actual row passed to Rubin injection.

`ulens_lsst` passes a **magnitude**, not a precomputed flux, in the injection catalog. `VisitInjectTask` receives that catalog plus the Exposure's PSF, PhotoCalib, and WCS. Rubin's object generator converts the catalog magnitude using the supplied PhotoCalib.

In [ ]:
expected_inst_flux = float(photo_calib.magnitudeToInstFlux(REQUESTED_MAG))
display(pd.Series({
    "requested_mag": REQUESTED_MAG,
    "expected_inst_flux": expected_inst_flux,
    "flux semantics": "instrumental flux in this calibrated Exposure's image scale",
}, name="explicit PhotoCalib conversion"))

In [ ]:
from ulens_lsst.lsst_tools import LSSTTools

tools = LSSTTools(
    image,
    injection_config={"noise_seed": INJECTION_NOISE_SEED},
)
injection_catalog = tools.create_injection_catalog(
    ra_values=[source_ra],
    dec_values=[source_dec],
    mag_values=[REQUESTED_MAG],
    magnification_values=[1.0],
    event_ids=[1],
    expMidptMJD=context.mjd,
    visit=context.visit,
    detector=context.detector,
)
display(injection_catalog.to_pandas().T.rename(columns={0: "injection row"}))

### What receives what?

- The catalog column passed to `VisitInjectTask` is `mag`; no flux column is supplied by this project.
- `LSSTTools.inject_sources()` passes `self.photo_calib`, which was obtained from `self.exposure.getPhotoCalib()`.
- It likewise passes the Exposure PSF and WCS.
- `VisitInjectTask.run()` temporarily attaches those objects to the cloned input Exposure and passes the supplied PhotoCalib and WCS to Rubin's GalSim object generator.
- The current processor does not supply a visit-summary calibration or override the magnitude conversion. It uses the already-loaded Exposure calibration.

Implementation to read next: `ulens_lsst/lsst_tools.py`, methods `_setup_injection_task`, `create_injection_catalog`, and `inject_sources`.

## 6. Inspect the injection configuration

**What are we trying to understand?** What Rubin is configured to change.

The current workflow starts from an already calibrated image. It may add source signal, source shot noise, variance, and mask bits. It does not run a photometric-calibration estimation task.

In [ ]:
inject_config = tools.injection_task.config
config_names = [
    "add_noise",
    "inject_variance",
    "noise_seed",
    "mask_plane_name",
    "calib_flux_radius",
    "bad_mask_names",
    "external_psf",
    "external_photo_calib",
    "external_wcs",
]
display(pd.Series({name: getattr(inject_config, name) for name in config_names}, name="VisitInjectConfig"))
print("PSF object passed from Exposure:", tools.psf is exposure.getPsf())
print("PhotoCalib object passed from Exposure:", tools.photo_calib is exposure.getPhotoCalib())
print("WCS object passed from Exposure:", tools.wcs is exposure.getWcs())

Interpretation:

- `add_noise=True` adds source Poisson noise rather than only a noiseless model.
- `inject_variance=True` updates the variance consistently with injected signal/noise.
- `mask_plane_name='INJECTED'` identifies affected pixels; Rubin also manages an injection-core plane.
- The passed PSF renders the point source; the passed PhotoCalib converts magnitude; the passed WCS places it.
- The `external_*` fields are Pipeline connection controls. In this direct task call, the actual `run()` arguments shown above are authoritative.

## 7. Inject in memory and test PhotoCalib preservation

**Question:** does injection recompute or modify the photometric calibration?

**Expected in the current workflow:** no. `LSSTTools` clones the calibrated Exposure, and `VisitInjectTask` injects pixels using the supplied calibration. It is not a calibration-fitting task.

In [ ]:
photo_before = exposure.getPhotoCalib()
zero_flux_before = float(photo_before.getInstFluxAtZeroMagnitude())
test_flux_before = float(photo_before.magnitudeToInstFlux(REQUESTED_MAG))

injection_result = tools.inject_sources(injection_catalog)
injected_exposure = injection_result["output_exposure"]
output_injection_catalog = injection_result["output_catalog"]

photo_after = injected_exposure.getPhotoCalib()
zero_flux_after = float(photo_after.getInstFluxAtZeroMagnitude())
test_flux_after = float(photo_after.magnitudeToInstFlux(REQUESTED_MAG))

display(pd.DataFrame({
    "before": [zero_flux_before, test_flux_before],
    "after": [zero_flux_after, test_flux_after],
    "difference": [zero_flux_after-zero_flux_before, test_flux_after-test_flux_before],
}, index=["instFlux at magnitude zero", f"instFlux at magnitude {REQUESTED_MAG:g}"]))
print("Same PhotoCalib Python object:", photo_before is photo_after)
print("Numerically identical conversion:", test_flux_before == test_flux_after)

**Interpretation:** in this direct current workflow, injection modifies an in-memory clone's pixels/variance/masks but does not recompute PhotoCalib. This conclusion applies to injection into an already processed calibrated image. It must not be generalized to a hypothetical pipeline that injects sources before photometric calibration is estimated.

**A problem would be:** changed PhotoCalib conversion values, or evidence that a calibration-fitting task ran.

## 8. Inspect pixels, variance, and mask before and after

**What are we trying to understand?** The concrete image-level effect of `VisitInjectTask`.

In [ ]:
original_image_cutout = cutout(exposure.image.array, source_x, source_y)
injected_image_cutout = cutout(injected_exposure.image.array, source_x, source_y)
difference_cutout = injected_image_cutout - original_image_cutout

vmin, vmax = np.nanpercentile(original_image_cutout, [2, 98])
dmax = np.nanmax(np.abs(difference_cutout))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(original_image_cutout, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Before")
axes[1].imshow(injected_image_cutout, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("After")
im = axes[2].imshow(difference_cutout, origin="lower", cmap="coolwarm", vmin=-dmax, vmax=dmax)
axes[2].set_title("After - before")
for axis in axes:
    axis.set_xlabel("Cutout x (pixel)")
    axis.set_ylabel("Cutout y (pixel)")
fig.colorbar(im, ax=axes[2], label="Instrumental image units")
fig.tight_layout()
plt.show()

In [ ]:
original_variance = cutout(exposure.variance.array, source_x, source_y)
injected_variance = cutout(injected_exposure.variance.array, source_x, source_y)
original_mask = cutout(exposure.mask.array, source_x, source_y)
injected_mask = cutout(injected_exposure.mask.array, source_x, source_y)

display(pd.Series({
    "sum(image after - before) in square cutout": float(np.sum(difference_cutout)),
    "sum(variance after - before)": float(np.sum(injected_variance-original_variance)),
    "pixels whose mask bits changed": int(np.count_nonzero(injected_mask != original_mask)),
    "output injection_flag": int(output_injection_catalog["injection_flag"][0]),
}, name="local injection changes"))

from ulens_lsst.scientific_validation import image_delta_diagnostic
delta_diagnostic = image_delta_diagnostic(
    exposure, injected_exposure, source_x, source_y, radius_pixels=12
)
display(pd.Series(delta_diagnostic, name="scientific-validation image-delta diagnostic"))

The circular local delta sum should have the correct sign and approximate flux scale. It is a diagnostic, not the definitive PSF flux: finite aperture size, PSF wings, edge clipping, masks, and injected noise can all affect a pixel sum. Forced PSF measurement is the primary photometric comparison.

## 9. Forced measurement before injection

**Question:** what flux did the real scene already contain at this position?

We run the same configured `ForcedMeasurementTask` at the known coordinate on the original image. A random-looking location can still have non-zero flux because of sky residuals, noise, unresolved sources, or PSF overlap from neighboring scene objects.

In [ ]:
pre_table = tools.measure_photometry(
    exposure,
    injection_catalog,
    include_native_diagnostics=True,
)
pre_row = pre_table[0]
pre_meas_flux = float(pre_row["meas_flux"])
pre_meas_flux_err = float(pre_row["meas_flux_err"])
pre_native_flags = {
    name: pre_row[name]
    for name in pre_table.colnames
    if name.startswith("native_")
}
display(pd.Series({
    "pre_meas_flux": pre_meas_flux,
    "pre_meas_flux_err": pre_meas_flux_err,
    "measure_flag": str(pre_row["measure_flag"]),
    **{name: bool(value) if isinstance(value, (bool, np.bool_)) else value for name, value in pre_native_flags.items()},
}, name="before injection"))

## 10. Forced measurement after injection and explicit recovery arithmetic

**Expected:** the post-injection forced flux is approximately the pre-existing scene flux plus the injected source contribution.

In [ ]:
post_table = tools.measure_photometry(
    injected_exposure,
    output_injection_catalog,
    include_native_diagnostics=True,
)
post_row = post_table[0]
post_meas_flux = float(post_row["meas_flux"])
post_meas_flux_err = float(post_row["meas_flux_err"])
post_native_flags = {
    name: post_row[name]
    for name in post_table.colnames
    if name.startswith("native_")
}
display(pd.Series({
    "post_meas_flux": post_meas_flux,
    "post_meas_flux_err": post_meas_flux_err,
    "measure_flag": str(post_row["measure_flag"]),
    **{name: bool(value) if isinstance(value, (bool, np.bool_)) else value for name, value in post_native_flags.items()},
}, name="after injection"))

In [ ]:
# Keep the arithmetic visible: these are the quantities used by scientific validation.
recovered_injected_flux = post_meas_flux - pre_meas_flux
flux_residual = recovered_injected_flux - expected_inst_flux
fractional_flux_residual = flux_residual / expected_inst_flux

display(pd.Series({
    "pre_meas_flux": pre_meas_flux,
    "post_meas_flux": post_meas_flux,
    "post - pre = recovered_injected_flux": recovered_injected_flux,
    "PhotoCalib expected_inst_flux": expected_inst_flux,
    "recovered - expected = flux_residual": flux_residual,
    "flux_residual / expected = fractional_flux_residual": fractional_flux_residual,
}, name="explicit recovery calculation"))

### Exact current meaning of `meas_flux`

`LSSTTools.measure_photometry()` copies `base_PsfFlux_instFlux` from the forced-measurement catalog into `meas_flux`. In production it is measured only on the post-injection Exposure. Therefore it is **total post-injection forced PSF instrumental flux**, not an artificial-source-only difference flux.

The numerical evidence above is that `post_meas_flux` and `post_meas_flux - pre_meas_flux` differ whenever the original scene measurement is non-zero. Production does not currently compute or persist `pre_meas_flux`; validation does so specifically to isolate the added source.

## 11. Audit historical and current flux columns

The historical production schema in `simulation_pipeline.py` wrote `ideal_mag`, measured magnitude/flux/error, magnification, and flags. It did **not** define `injected_flux`. Some historical tutorial DataFrames added an `injected_mag` display column, but that was not the historical production Parquet schema.

The new PROCESS normalization adds `injected_mag = ideal_mag` and creates `injected_flux = NaN`. That NaN is a current normalization placeholder: it is not returned by Rubin and is not evidence that historical instrumental injected flux was lost during the refactor—historical production did not persist it either.

In [ ]:
flux_audit = pd.DataFrame([
    ["ideal_mag", "light_curves.py; processor input/catalog output", "instantaneous model magnitude requested for this epoch", "mag", "yes, as catalog `mag`", "historical and new", "yes"],
    ["injected_mag", "new _scientific_rows normalization", "alias of ideal_mag, not an independently measured quantity", "mag", "indirectly", "new path; historical tutorials only", "yes, from ideal_mag"],
    ["injected_flux", "new _scientific_rows normalization", "currently an unpopulated placeholder (NaN)", "intended instrumental flux", "no", "new path as NaN", "not from Parquet alone"],
    ["expected_inst_flux", "scientific_validation.py", "PhotoCalib.magnitudeToInstFlux(requested_mag)", "Exposure instrumental flux", "Rubin computes equivalent internally", "validation product only", "requires original PhotoCalib"],
    ["meas_flux", "LSSTTools.measure_photometry", "post-injection base_PsfFlux_instFlux", "Exposure instrumental flux", "no", "historical and new", "already stored"],
    ["meas_flux_err", "LSSTTools.measure_photometry", "Rubin PSF instrumental-flux uncertainty", "Exposure instrumental flux", "no", "historical and new", "already stored"],
], columns=["Quantity", "Created where", "Meaning", "Units", "Passed to injection?", "Persisted?", "Reconstructible later?"])
display(flux_audit.set_index("Quantity"))

**Audit answers:**

- Historical production Parquet stored the injected **magnitude input** as `ideal_mag`, but not calibrated instrumental injected flux.
- Exact injected instrumental flux cannot be reconstructed from the photometry Parquet alone. One must reload the image (or have separately persisted PhotoCalib information) and apply the same PhotoCalib.
- Current `injected_flux = NaN` is a placeholder/normalization artifact. The schema has not yet been changed because preserving calibrated flux and/or pre-injection photometry is a scientific production decision.

## 12. Inspect one current production processor row

**What are we trying to understand?** What the `InjectionForcedPhotometryProcessor` returns and what PROCESS adds before Parquet persistence.

This repeats one tiny in-memory injection with the same deterministic injection configuration. It does not write a run product.

In [ ]:
from ulens_lsst.image_processors import get_image_processor
from ulens_lsst.image_processing_run import _scientific_rows

source_frame = pd.DataFrame([{
    "event_id": 1,
    "ra": source_ra,
    "dec": source_dec,
    "ideal_mag": REQUESTED_MAG,
    "magnification": 1.0,
}])
processor = get_image_processor("injection_forced_photometry")
processor_result = processor.process(
    image,
    context,
    source_frame,
    config={"injection_config": {"noise_seed": INJECTION_NOISE_SEED}},
)
production_row = _scientific_rows(
    processor_result, source_frame, context, run_id="guided-notebook"
).iloc[0]
display(production_row.to_frame("persisted value"))

Column groups in this row:

- **Identity:** `event_id`, `image_id`, visit, detector, band, `injection_id`.
- **Truth/request:** `ideal_mag`, magnification, and the alias `injected_mag`.
- **Measurement:** `meas_flux`, `meas_flux_err`, measured magnitude/error.
- **Scientific status:** eligibility, attempt, injection/measurement/processing status, and compact flags.
- **Placeholder:** `injected_flux` is NaN.

After a RAC run, the post-injection measurement and magnitude request remain. Without revisiting the Rubin image, the current row loses the original-scene forced flux, exact PhotoCalib-derived expected flux, detailed calibration object, pixel-level delta, and validation-only native diagnostics.

## 13. Compare the manual arithmetic with the validation API

**Expected:** the independently visible manual values above agree with `run_controlled_validation()` for the same source, image, and Rubin noise seed.

In [ ]:
from ulens_lsst.scientific_validation import (
    ControlledValidationSource,
    run_controlled_validation,
)

validation_source = ControlledValidationSource(
    source_id="guided-source",
    ra_deg=source_ra,
    dec_deg=source_dec,
    magnitude=REQUESTED_MAG,
    seed=INJECTION_NOISE_SEED,
)
validation_row = run_controlled_validation(
    image,
    context,
    [validation_source],
    release_name=release_config.name,
    injection_noise_seed=INJECTION_NOISE_SEED,
)[0]

comparison = pd.DataFrame({
    "manual": [expected_inst_flux, pre_meas_flux, post_meas_flux, recovered_injected_flux, fractional_flux_residual],
    "validation API": [
        validation_row["expected_inst_flux"],
        validation_row["pre_meas_flux"],
        validation_row["post_meas_flux"],
        validation_row["recovered_injected_flux"],
        validation_row["fractional_flux_residual"],
    ],
}, index=["expected_inst_flux", "pre_meas_flux", "post_meas_flux", "recovered_injected_flux", "fractional_flux_residual"])
comparison["difference"] = comparison["validation API"] - comparison["manual"]
display(comparison)
np.testing.assert_allclose(comparison["manual"], comparison["validation API"], rtol=1e-12, atol=1e-12)
print("Manual and validation calculations agree.")

## 14. Understand the uncertainty/pull limitation

The current diagnostic combines the two reported errors as though pre/post measurements were independent. They are not: both measurements share the same underlying real-image pixels.

In [ ]:
naive_sigma_delta = np.sqrt(pre_meas_flux_err**2 + post_meas_flux_err**2)
naive_pull = flux_residual / naive_sigma_delta
display(pd.Series({
    "pre_err": pre_meas_flux_err,
    "post_err": post_meas_flux_err,
    "sqrt(pre_err² + post_err²)": naive_sigma_delta,
    "residual / naive_sigma_delta": naive_pull,
}, name="naive pull calculation"))

The correct variance identity is

$$
\mathrm{Var}(F_{post}-F_{pre}) = \mathrm{Var}(F_{post}) + \mathrm{Var}(F_{pre}) - 2\,\mathrm{Cov}(F_{post}, F_{pre}).
$$

The shared scene noise makes the covariance non-zero. `ulens_lsst` does not yet have a validated covariance solution. Therefore the sub-unit pull widths in the current validation record cannot be interpreted directly as Rubin uncertainty miscalibration.

## 15. Revisit the persisted density experiment

This section reads the small maintained Parquet product; it does not rerun injection. The artifact is selected by its provenance `release` field, not by a release-to-path Butler mapping.

In [ ]:
import pyarrow.parquet as pq

validation_root = REPO_ROOT / "docs/_static/scientific_validation/2026-09-25"
density_path = None
for provenance_path in validation_root.glob("*/provenance.json"):
    provenance = json.loads(provenance_path.read_text())
    if provenance["release"] == release_config.name:
        density_path = provenance_path.parent / "density-validation.parquet"
        break
if density_path is None:
    raise FileNotFoundError(f"No maintained density validation artifact for {release_config.name}.")

density = pq.read_table(density_path).to_pandas()
display(pd.DataFrame({
    "source count": sorted(density["n_injected_sources"].unique()),
    "density (sources / arcmin²)": sorted(density["injection_density_per_arcmin2"].unique()),
}))
print("Reference source IDs:", sorted(density["source_id"].unique()))
display(density[[
    "n_injected_sources", "source_id", "requested_mag",
    "fractional_flux_residual", "fractional_change_from_low_density",
    "injection_failure_count", "measurement_failed",
]].head(12))

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4))
for magnitude, group in density.groupby("requested_mag"):
    axis.scatter(
        group["injection_density_per_arcmin2"],
        group["fractional_change_from_low_density"],
        label=f"mag {magnitude:g}",
        alpha=0.75,
    )
axis.axhline(0, color="black", linewidth=0.8)
axis.set_xlabel("Injection density (sources / arcmin²)")
axis.set_ylabel("Same-source fractional flux change from lowest density")
axis.legend()
axis.set_title(f"Maintained density validation: {release_config.name}")
plt.show()

**What this demonstrates:** for the tested image, magnitudes, and positions, adding well-separated artificial sources through the tested density did not measurably change recovered flux for the fixed reference sources.

**What it does not demonstrate:** behavior for close blends, a universal safe maximum, behavior across many images/conditions, or recalibration of PhotoCalib. The explicit test above addresses the current direct injection workflow's calibration object; the density experiment is about photometric self-contamination with separated sources.

## 16. The scientific-validation status document

After completing this notebook, read [`docs/scientific_validation_status.rst`](../scientific_validation_status.rst).

Interpret its status labels narrowly:

- **PASS:** the stated question passed in the explicitly stated regime.
- **INITIAL:** evidence exists, but the sample/conditions are limited.
- **INCONCLUSIVE:** the current method or sample cannot support a conclusion.
- **NOT YET TESTED:** no maintained evidence exists yet.

The table is deliberately not a completeness or detection-efficiency claim.

In [ ]:
validation_status_path = REPO_ROOT / "docs/scientific_validation_status.rst"
print(validation_status_path)
print("Exists:", validation_status_path.exists())

## 17. Computational execution model

The scientific image is the basic processing unit; a deterministic execution work unit may contain one or more image IDs.

```text
PREPARE
    -> events/
    -> images.parquet
    -> work_units.parquet

PROCESS(work_unit)
    -> bounded event/image reads
    -> Butler image load
    -> selected ImageProcessor
    -> independent photometry/part-<work-unit>.parquet
    -> independent status/part-<work-unit>.json

STATUS / RESTART
    -> classify deterministic work-unit state

FINALIZE
    -> validate parts, schemas, completeness, and duplicate keys
       without concatenating the campaign into pandas
```

In [ ]:
from ulens_lsst.work_units import build_work_unit_rows, work_unit_product_path

example_work_unit = build_work_unit_rows([context.image_id], images_per_work_unit=1)[0]
example_status = {
    "work_unit_id": example_work_unit["work_unit_id"],
    "state": "completed",
    "n_images": 1,
    "n_photometry_rows": 1,
    "elapsed_seconds": 4.2,
    "output_path": str(work_unit_product_path("RUN_DIR", "photometry", example_work_unit["work_unit_id"])),
}
display(pd.Series(example_work_unit, name="example work_units.parquet row"))
display(pd.Series(example_status, name="example independent status record"))

- Work-unit IDs hash sorted scientific image IDs; they do not depend on worker number, CPU count, execution order, retry, or SLURM job ID.
- Each work unit owns one deterministic Parquet part, avoiding concurrent append and a global DataFrame.
- A worker writes a temporary file, validates it, and atomically replaces its final part. A process killed before rename leaves no valid completed part.
- STATUS combines expected work units, independent status records, and output validation.
- RESTART selects missing, failed, stale, or corrupt work while preserving valid completed parts.
- FINALIZE scans bounded keys/metadata and uses scalable duplicate checking rather than `pd.concat()` of all science rows.

### Optional tiny campaign commands

These commands are safe only after PREPARE has produced a tiny run directory containing `config_file.yaml`, `events/`, `images.parquet`, and `work_units.parquet`. The current CLI exposes PROCESS/STATUS/RESTART/FINALIZE directly; PREPARE is currently entered through the release-aware `SimPipeline` simulation path rather than a standalone `prepare` subcommand.

```bash
# Inspect the first deterministic work-unit ID from work_units.parquet, then:
ulens_lsst process TINY_RUN_DIR WORK_UNIT_ID
ulens_lsst status TINY_RUN_DIR
ulens_lsst restart-plan TINY_RUN_DIR
ulens_lsst finalize TINY_RUN_DIR
```

Do not point these commands at a production run while learning. The in-memory single-image cells above are sufficient to understand the science path without writing campaign outputs.

## 18. Where to read the implementation

- [`ulens_lsst/release_config.py`](../../ulens_lsst/release_config.py): `resolve_data_release`, processor/release compatibility boundary.
- [`ulens_lsst/lsst_data.py`](../../ulens_lsst/lsst_data.py): `LSSTData.__init__`, `select_image_contexts`, `load_image`, and the `Calexp` compatibility wrapper.
- [`ulens_lsst/image_context.py`](../../ulens_lsst/image_context.py): `ImageContext` and generic image identity/context.
- [`ulens_lsst/image_processors.py`](../../ulens_lsst/image_processors.py): `ImageProcessor`, `InjectionForcedPhotometryProcessor`, and the static registry.
- [`ulens_lsst/lsst_tools.py`](../../ulens_lsst/lsst_tools.py): injection catalog creation, `VisitInjectTask`, and `ForcedMeasurementTask` setup/execution.
- [`ulens_lsst/image_processing_run.py`](../../ulens_lsst/image_processing_run.py): `process_work_unit`, bounded reads, row normalization, and atomic scientific output.
- [`ulens_lsst/work_units.py`](../../ulens_lsst/work_units.py): deterministic image/injection/work-unit identities and output paths.
- [`ulens_lsst/run_state.py`](../../ulens_lsst/run_state.py): STATUS, restart selection, part validation, and FINALIZE.
- [`ulens_lsst/scientific_validation.py`](../../ulens_lsst/scientific_validation.py): controlled before/after validation and validation schema.
- [`ulens_lsst/density_validation.py`](../../ulens_lsst/density_validation.py): nested density experiment.
- [`ulens_lsst/validation_analysis.py`](../../ulens_lsst/validation_analysis.py): summaries and diagnostic plots.

## 19. Questions I should now be able to answer

- [ ] What is an `ImageContext`?
- [ ] How does `DataReleaseConfig` isolate DP0.2/DP1 differences?
- [ ] What exactly is passed to `VisitInjectTask`?
- [ ] What does PhotoCalib do in this experiment?
- [ ] Does injection recompute PhotoCalib?
- [ ] What pixels, variance, and masks does injection change?
- [ ] What exactly does `meas_flux` mean?
- [ ] Why can `meas_flux` contain pre-existing scene flux?
- [ ] Why do we measure before and after injection during validation?
- [ ] What is `expected_inst_flux`?
- [ ] What is actually stored in production `photometry/`?
- [ ] Was injected flux already stored historically?
- [ ] What information would be lost after a RAC run?
- [ ] What does the current density experiment prove?
- [ ] What does it not prove?
- [ ] Why are the current pull widths not yet an uncertainty-calibration result?
- [ ] What is the difference between forced-photometry recovery and source/detection completeness?

The goal is to answer these in scientific language, not to memorize class names.